In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import hmean
import matplotlib.pyplot as plt

In [ ]:
plt.style.use("default")

figures_dir = 'figures'


def save_figure(fig, name):
    """Export a figure, as a 300 dpi PNG for previews and as a vector PDF for the post."""
    os.makedirs(figures_dir, exist_ok=True)
    for ext, dpi in (('png', 300), ('pdf', None)):
        path = os.path.join(figures_dir, f'{name}.{ext}')
        fig.savefig(path, dpi=dpi, bbox_inches='tight')
        print('saved', path)

In [ ]:
'''Load ability profiles'''
nature_profiles_df = pd.read_csv("data/ability_profiles_MaxAuxDiff=0_DupThresh=100.csv", index_col=0)
additional_profiles_df = pd.read_csv('data/additional_results_16June2026.csv', index_col=0)
ability_profiles_df = pd.concat([nature_profiles_df, additional_profiles_df], axis=0)

In [ ]:
'''Compute reduced profiles with 1 score for each of 10 group of dimensions instead of 1 score for each of 18 dimensions'''
#We use the fact that each dimension code starts with the exact two letters that make up the code of the group it belongs to. Group SN is treated separately due to being the only group that contains exactly one dimension but with a third letter in the dimension code.
dimension_groups = list(set([name[:2] for name in ability_profiles_df.columns]))
dimension_groups.sort()
dimensions = list(ability_profiles_df.columns)
for dimension_group in dimension_groups:
    components = [dimension for dimension in dimensions if dimension[:2] == dimension_group]
    if len(components) >= 2:
        ability_profiles_df[dimension_group] = ability_profiles_df[components].apply(hmean, axis=1)
ability_profiles_df['SN'] = ability_profiles_df['SNs'].copy()
reduced_profiles_columns = list(dimension_groups)
reduced_profiles_df = ability_profiles_df[reduced_profiles_columns].copy()

In [ ]:
'''Functions used to make radar plots. Mostly inherited from the AeLe github repo (https://github.com/Kinds-of-Intelligence-CFI/ADeLe-AIEvaluation/blob/main/ability_profiles/scc_and_ability_profiles.ipynb) , with a few edits.'''
def plot_radar_auc_subplots(auc_dict, feature_names, name=None, title="AUCs from Logistic Curves", level_diff=0, dup_threshold=100, max_level=9):
    """
    Create a radar plot comparing models using the area under their estimated logistic curves.
    """
    '''desired_order = [
        "AS", "CEc", "CEe", "CL", "MCr", "MCt", "MCu", "MS",
        "QLl", "QLq", "SNs", "KNa", "KNc", "KNf", "KNn", "KNs",
        "AT", "VO"
    ]'''
    #desired_order = order
    #feature_names = [feat for feat in desired_order if feat in feature_names]
    N = len(feature_names)

    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1] #Pourquoi ?

    fig, axes = plt.subplots(2, 2, figsize=(20, 20), subplot_kw=dict(polar=True), constrained_layout=True)
    for i in range(2):
        for j in range(2):
            axes[i][j].set_theta_direction(-1)
            axes[i][j].set_theta_offset(np.pi/2)

    def plot_single_radar(ax, models_group, group_name):
        for model in models_group:
            display_name = name_mapping.get(model, model)
            values = [auc_dict[model][feat] for feat in feature_names] # donc : auc_dict : dictionnaire contenant un dictionnaire par modèle. Chaque sous-dictionnaire est un dictionnaire de features. Exemple minimal : {'GPT-4o' : {'AS' : 3.74}}
            values = [0 if np.isnan(v) else v for v in values]
            values += values[:1]

            if group_name == 'gpt_o1':
                alpha = dict(gpt_o1_models)[model]
                base_color = plt.cm.Reds(alpha)
            elif group_name == 'llama':
                alpha = dict(llama_models)[model]
                base_color = plt.cm.Blues(alpha)
            elif group_name == 'ds_r1':
                alpha = dict(ds_r1_models)[model]
                base_color = plt.cm.Greens(alpha)
            elif group_name == 'gemini':
                alpha = dict(gemini_models)[model]
                base_color = plt.cm.Purples(alpha)

            ax.plot(angles, values, label=display_name, color=base_color, linewidth=2)
            ax.fill(angles, values, alpha=0.25, color=base_color)

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(feature_names, fontsize=17)
        ax.tick_params(axis='x', pad=18)
        ax.set_ylim(0, max_level)
        ax.tick_params(axis='y', labelsize=16)
        ncol = 2 if len(models_group) > 4 else 1
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05),
              fontsize=18, ncol=2, bbox_transform=ax.transAxes)
        #ax.legend(loc='upper right',  bbox_to_anchor=(1.02, 1.05), fontsize=16, ncol=2)

    plot_single_radar(axes[0][0], [model[0] for model in gpt_o1_models], 'gpt_o1')
    plot_single_radar(axes[0][1], [model[0] for model in llama_models], 'llama')
    plot_single_radar(axes[1][0], [model[0] for model in ds_r1_models], 'ds_r1')
    plot_single_radar(axes[1][1], [model[0] for model in gemini_models], 'gemini')

    #plt.tight_layout()
    #plt.subplots_adjust(top=0.85)
    if name:
        save_figure(fig, name)
    plt.show()

In [ ]:
gpt_o1_models = [
    ('babbage-002', 0.1),
    ('davinci-002', 0.2),
    ('gpt-35-turbo', 0.3),
    ('gpt4o', 0.4),
    ('o1-mini', 0.56),
    ('o1_re=low', 0.72),
    ('GPT-5.2-Chat', 0.88),
    ('OpenAI o3-mini', 1.0)
]

llama_models = [
    ('llama3d2-1b', 0.1),
    ('llama3d2-3b', 0.2),
    ('llama3d2-11b', 0.4),
    ('llama3d2-90b', 0.6),
    ('llama3d1-405b', 0.8),
    ('LLaMA-4-17B-128E', 1.0)
]

ds_r1_models = [
    ('DK-R1-Dist-Qwen-1.5B', 0.25),
    ('DK-R1-Dist-Qwen-7B', 0.50),
    ('DK-R1-Dist-Qwen-14B', 0.75),
    ('DK-R1-Dist-Qwen-32B', 1.0)
]

gemini_models = [
    ('Gemini-2.5-Flash', 0.33),
    ('Gemini-3.1-Flash', 0.66),
    ('Gemini-3.1-Pro', 1.0)
]

# Standardise the names
name_mapping = {
    'babbage-002': 'Babbage-002',
    'davinci-002': 'Davinci-002',
    'gpt-35-turbo': 'GPT-3.5-Turbo',
    'gpt4o': 'GPT-4o',
    'o1-mini': 'OpenAI o1-mini',
    'o1_re=low': 'OpenAI o1',
    'llama3d2-1b': 'LLaMA-3.2-1B-Instruct',
    'llama3d2-3b': 'LLaMA-3.2-3B-Instruct',
    'llama3d2-11b': 'LLaMA-3.2-11B-Instruct',
    'llama3d2-90b': 'LLaMA-3.2-90B-Instruct',
    'llama3d1-405b': 'LLaMA-3.1-405B-Instruct',
    'DK-R1-Dist-Qwen-1.5B': 'DK-R1-Dist-Qwen-1.5B',
    'DK-R1-Dist-Qwen-7B': 'DK-R1-Dist-Qwen-7B',
    'DK-R1-Dist-Qwen-14B': 'DK-R1-Dist-Qwen-14B',
    'DK-R1-Dist-Qwen-32B': 'DK-R1-Dist-Qwen-32B',
    'Gemini-2.5-Flash': 'Gemini-2.5-Flash',
    'Gemini-3.1-Flash': 'Gemini-3.1-Flash',
    'Gemini-3.1-Pro': 'Gemini-3.1-Pro',
    'LLaMA-4-17B-128E': 'LLaMA-4-17B-128E',
    'GPT-5.2-Chat': 'GPT-5.2-Chat',
    'OpenAI o3-mini': 'OpenAI o3-mini'
}

In [ ]:
reduced_dict = reduced_profiles_df.transpose().to_dict()

In [ ]:
reciprocal_name_mapping = {v : k for k, v in name_mapping.items()}

In [ ]:
reduced_dict = {reciprocal_name_mapping[k] : reduced_dict[k] for k in reduced_dict.keys()}

In [ ]:
full_profile_dict = ability_profiles_df.transpose().to_dict()
full_profile_dict = {reciprocal_name_mapping[k]: full_profile_dict[k] for k in full_profile_dict.keys()}

In [ ]:
plot_radar_auc_subplots(full_profile_dict, name='radar_18_dimensions', feature_names=[
        "AS", "CEc", "CEe", "CL", "MCr", "MCt", "MCu", "MS",
        "QLl", "QLq", "SNs", "KNa", "KNc", "KNf", "KNn", "KNs",
        "AT", "VO"
    ])

In [ ]:
plot_radar_auc_subplots(reduced_dict, name='radar_10_groups', feature_names=['AS','CE', 'CL', 'MC', 'MS', 'QL', 'SN', 'KN', 'AT', 'VO'])